# EMA 6611 – Optoelectronics Materials Processing (Fall 2026)
## Homework 3: Passivation and Optical Surfaces and Coatings — Writing the Report as You Go

**Assigned:** Monday, October 26, 2026

**Due:** Sunday, November 8, 2026, 11:59 PM ET (submit on Webcourses)

**Total: 100 points** (+5 bonus)

---

**Name:** *(double-click to edit)*

**UCF ID:** *(double-click to edit)*

> **Before you start:** you opened this notebook from the course GitHub repository, which is read-only. Go to **File → Save a copy in Drive** *right now* so you are working in your own copy — otherwise your work will not be saved.


## Overview and Learning Objectives

Homework 2 covered the absorber or emitter and the carrier-selective junctions. This assignment covers the two building blocks that decide how many of the generated (or injected) carriers survive and how many of the photons get in or out: **passivation** — bulk and surface recombination, and the processes that suppress them — and **optical surfaces and coatings** — texture, antireflection coatings and reflectors. As in Homework 2, each task pairs an analysis in this notebook (calculators for lifetime, surface recombination, reflectance and coating design that you feed with the numbers of *your* material and device) with the corresponding subsection of the Discussion and Analysis written in Overleaf while the analysis is fresh. By the end you will have drafted the **Passivation** and **Optical Surfaces and Coatings** subsections of your research report.

After completing this assignment, you will be able to:

1. Compute the radiative and Auger limits to carrier lifetime for your material and injection level, compare them with the lifetimes reported in your corpus, and identify which mechanism limits the bulk.
2. Describe the bulk and surface passivation strategies used for your device, including the deposition and post-deposition processes, and extract $S$, $J_{0s}$, $D_{it}$ and $Q_f$ ranges from the literature where they exist.
3. Use an extended Shockley–Read–Hall surface model to show how fixed charge and interface-state density set the surface recombination velocity, and combine bulk and surface terms into an effective lifetime and an implied open-circuit voltage.
4. Compute the Fresnel reflectance of your bare material, the escape cone of an emitter, and the spectrally weighted reflectance of single- and double-layer antireflection coatings and rear reflectors with the transfer-matrix method.
5. Identify the texturing, coating and reflector approaches used in your field, with their processing, and quantify the optical losses they address.

**Connection to your research report:** the two subsections drafted here are worth 10 of the report's 100 points (Table 1 of the *Research Report: Assignment and Rubric*); Table 2 of that document lists what each must cover, and the tasks below are organized to cover those items. Keep the process → structure → properties → performance framing in view: the calculators give you the properties → performance links; your corpus supplies the processes and structures.

**What you submit (four files, all on Webcourses):**

1. This notebook printed to PDF (`EMA-6611-HW03-LastName.pdf`).
2. The notebook file (`.ipynb`).
3. Your extended `survey.csv` (Tasks 1–4 add columns to the file from Homework 2).
4. **Your report compiled from Overleaf as of submission** (`LastName-Report-HW03.pdf`). The report subsections are graded from this PDF; the notebook is the supporting evidence.

**Grading:** Task 1 — 25 points; Task 2 — 25 points; Task 3 — 20 points; Task 4 — 30 points; bonus (Task 4.4) — up to 5 points. Each task states how its points divide between the notebook and the report.


## Ground Rules

* Work through the notebook top to bottom. Cells marked **`# YOUR CODE HERE`** and *Your answer here* are graded deliverables; the instructor-provided module cells are run as they are.
* **AI use** (per the course AI policy): you *may* use generative AI — including the Gemini assistant built into Colab — for code (for example, "plot the effective lifetime for an n-type wafer with J0s = 5 fA/cm² on both sides"), for debugging, and for finding and summarizing literature. You must be able to explain every line of code you submit. Do **not** use AI to write the prose in this notebook or in your report. Disclose all AI use in the disclosure cell at the end.
* Every number you take from an article, datasheet or database must be cited in the report with `\cite{}` and must appear in `references.bib`. The calculators below ship with *starting-point* material parameters; replace them with values from your corpus and cite those.
* Discussing concepts with classmates is encouraged; the work you submit must be your own.
* Before submitting, confirm the whole notebook runs cleanly: **Runtime → Restart and run all**.


## Setup

Run the three cells below first. The second installs `pvlib` (for the AM1.5G spectrum) and `pyyaml`; the setup must be re-run each time the runtime restarts.


In [ ]:
# Imports and plotting configuration — do not modify
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import constants

plt.rcParams.update({
    'figure.figsize': (7, 4.5),
    'figure.dpi': 110,
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
    # course figure style: tick marks inside, mirrored on the top and right axes, larger
    'xtick.direction': 'in', 'ytick.direction': 'in',
    'xtick.top': True, 'ytick.right': True,
    'xtick.major.size': 6, 'ytick.major.size': 6,
    'xtick.minor.size': 3.5, 'ytick.minor.size': 3.5,
    'xtick.minor.visible': True, 'ytick.minor.visible': True,
})

# Physical constants (SI unless noted)
q   = constants.e            # elementary charge [C]
kB  = constants.k            # Boltzmann constant [J/K]
kB_eV = constants.k / constants.e   # Boltzmann constant [eV/K]
h   = constants.h            # Planck constant [J·s]
c   = constants.c            # speed of light [m/s]
eps0 = constants.epsilon_0   # vacuum permittivity [F/m]

print('Setup complete.')


In [ ]:
# Package installation — do not modify
%pip -q install pvlib pyyaml
import pvlib, yaml
print('Packages installed.')


In [ ]:
# Upload your survey.csv from Homework 2 to this session (folder icon at left → upload), then run this cell.
import os
assert os.path.exists('survey.csv'), 'survey.csv not found: upload the file from Homework 2 to this Colab session.'
survey = pd.read_csv('survey.csv')
print(f'survey.csv: {len(survey)} articles, {int((survey.top20.astype(str).str.lower()=="yes").sum())} in the top 20, {len(survey.columns)} columns.')


---
## Task 1 — Bulk recombination and bulk passivation (25 points)

Begins the **Passivation** subsection of the report. Points: 1.1 — 10 (notebook); 1.2 — 15 (5 notebook validation, 10 report).

### Background: three ways to lose a carrier

When light or injection adds excess carriers $\Delta n = \Delta p$, each recombination path removes them at its own rate and the rates add, $1/\tau_{bulk} = 1/\tau_{rad} + 1/\tau_{Auger} + 1/\tau_{SRH}$ (Grundmann, *The Physics of Semiconductors*, 4th ed., Ch. 10). Two of the three are intrinsic to the material: **radiative** recombination, $U_{rad} = B(np - n_i^2)$, with $B \sim 10^{-10}$ cm$^3$ s$^{-1}$ in direct-gap materials and $\sim 10^{-14}$ in silicon, and **Auger** recombination, $U_{Auger} = (C_n n + C_p p)(np - n_i^2)$, which takes over at high doping and high injection (for silicon the parameterization of Richter et al., Phys. Rev. B 86, 165202 (2012), is built in). The third, **Shockley–Read–Hall** recombination through deep levels (Shockley & Read, Phys. Rev. 87, 835 (1952); Hall, Phys. Rev. 87, 387 (1952)), is set by defects and therefore by processing:

$$\tau_{SRH}(\Delta n) = \frac{\tau_{p0}(n_0 + n_1 + \Delta n) + \tau_{n0}(p_0 + p_1 + \Delta n)}{n_0 + p_0 + \Delta n}, \qquad \tau_{n0} = \frac{1}{\sigma_n v_{th} N_t},\ \ n_1 = n_i e^{(E_t - E_i)/kT}.$$

A measured lifetime far below the intrinsic limit means SRH (or surfaces, Task 2) dominates, and the processing papers in your corpus are about closing that gap. For emitters the same three terms appear as the ABC model, $\eta_{int} = Bn^2/(An + Bn^2 + Cn^3)$, in which defects ($A = 1/\tau_{SRH}$) take the low-current side and Auger ($C$) gives the droop (Grundmann, Ch. 10).

### 1.1 Intrinsic limits vs. what your corpus reports (10 points)

**(a)** Collect from your top-20 articles every reported carrier lifetime (or, for emitters, IQE/EQE vs. current and any ABC coefficients): the value, the measurement method (QSSPC/PCD, TRPL, PL quantum yield, EL, fit to $J$–$V$ or EQE) and the sample it was measured on (bare wafer, passivated wafer, finished device, thin film on glass). Lifetimes measured by different methods on different samples are different quantities — a TRPL decay of a perovskite film on glass is dominated by its surfaces, a QSSPC lifetime of a symmetrically passivated wafer is $\tau_{eff}$ of the whole wafer — so say which each is.

**(b)** With the calculator below, plot $\tau_{rad}$, $\tau_{Auger}$ and, if you can estimate it, $\tau_{SRH}$ against injection level for your material at the doping used in your device. Use $B$, $C$ and $n_i$ from your own sources (the presets are starting points and must be cited or replaced). Then put the reported lifetimes on the same axes and answer: which mechanism limits the bulk of your device, by how much do the best reported lifetimes fall short of the intrinsic limit, and at what injection level does the device operate? For emitters, plot $\eta_{int}$ against current density with `led_iqe` and locate the operating current.


In [ ]:
# ============================================================
# HW03 MODULE A — recombination, lifetime and surface passivation calculators
# (instructor-provided; run, do not modify)
# ============================================================
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq

kT = 0.025852           # eV at 300 K
Vt = kT                 # thermal voltage, V
q_C = 1.602176634e-19   # C
eps0 = 8.8541878128e-14 # F/cm
vth = 1.0e7             # thermal velocity, cm/s (Si, 300 K; order of magnitude for all)

# Intrinsic recombination parameters at 300 K. Starting points only — replace with values from your corpus and cite them.
# Si: n_i Altermatt et al. 2003; B Trupke et al. 2003; Auger via Richter et al. 2012 (handled separately below).
# GaAs/InP: B from Varshni 1967 / Levinshtein 1996; C order of magnitude (Strauss 1993). InGaN: typical ABC values
# (Piprek, Phys. Status Solidi A 207, 2217 (2010)). CdTe: B ~1e-10 (Kanevce et al. 2017). MAPbI3: k2, k3 from
# Johnston & Herz, Acc. Chem. Res. 49, 146 (2016). n_i for the wide-gap materials is negligible.
RECOMB = {
    'Si':     dict(ni=9.65e9,  B=4.73e-15, Cn=None, Cp=None, Eg=1.12, note='Auger: Richter 2012 parameterization'),
    'GaAs':   dict(ni=2.1e6,   B=1.7e-10,  Cn=1e-30, Cp=1e-30, Eg=1.42),
    'InP':    dict(ni=1.3e7,   B=1.2e-10,  Cn=9e-31, Cp=9e-31, Eg=1.34),
    'InGaN':  dict(ni=1e-5,    B=2e-11,    Cn=2e-30, Cp=2e-30, Eg=2.8, note='QW active region; A typically 1e7 s^-1'),
    'CdTe':   dict(ni=1e6,     B=1e-10,    Cn=1e-29, Cp=1e-29, Eg=1.50),
    'MAPbI3': dict(ni=1e5,     B=1e-10,    Cn=1e-28, Cp=1e-28, Eg=1.60, note='k2 ~ 1e-10 cm^3/s, k3 ~ 1e-28 cm^6/s'),
}


def carrier_densities(ni, N, dtype, dn):
    """Equilibrium and excess densities for doping N (cm^-3) of type 'n' or 'p' at injection dn."""
    if dtype == 'n': n0, p0 = N, ni ** 2 / N
    else:            p0, n0 = N, ni ** 2 / N
    return n0 + dn, p0 + dn, n0, p0


def tau_radiative(ni, B, N, dtype, dn):
    n, p, n0, p0 = carrier_densities(ni, N, dtype, dn)
    return dn / (B * (n * p - ni ** 2))


def tau_auger(ni, N, dtype, dn, Cn=None, Cp=None, material=None):
    """Auger lifetime. For Si the Richter et al. (Phys. Rev. B 86, 165202, 2012) parameterization is used;
    otherwise U = (Cn n + Cp p)(np - ni^2)."""
    n, p, n0, p0 = carrier_densities(ni, N, dtype, dn)
    if material == 'Si' or (Cn is None and Cp is None):
        g_eeh = 1 + 13 * (1 - np.tanh((n0 / 3.3e17) ** 0.66))
        g_ehh = 1 + 7.5 * (1 - np.tanh((p0 / 7.0e17) ** 0.63))
        U = (n * p - ni ** 2) * (2.5e-31 * g_eeh * n0 + 8.5e-32 * g_ehh * p0 + 3.0e-29 * dn ** 0.92)
    else:
        U = (Cn * n + Cp * p) * (n * p - ni ** 2)
    return dn / U


def tau_srh(ni, N, dtype, dn, tau_n0, tau_p0, Et_minus_Ei=0.0):
    """Shockley–Read–Hall lifetime vs injection for one defect level at Et - Ei (eV) with capture time constants
    tau_n0 = 1/(sigma_n v_th N_t), tau_p0 = 1/(sigma_p v_th N_t) (Shockley & Read 1952; Hall 1952)."""
    n, p, n0, p0 = carrier_densities(ni, N, dtype, dn)
    n1 = ni * np.exp(Et_minus_Ei / kT); p1 = ni * np.exp(-Et_minus_Ei / kT)
    return (tau_p0 * (n0 + n1 + dn) + tau_n0 * (p0 + p1 + dn)) / (n0 + p0 + dn)


def bulk_lifetime_plot(material, N, dtype, tau_n0=None, tau_p0=None, Et_minus_Ei=0.0, dn=None, ax=None, label=None, **override):
    """tau_rad, tau_Auger, tau_SRH (if given) and the bulk total vs injection level. Returns a DataFrame."""
    p = dict(RECOMB[material]); p.update(override)
    dn = np.logspace(12, 18, 200) if dn is None else np.asarray(dn)
    out = pd.DataFrame({'dn': dn})
    out['tau_rad'] = tau_radiative(p['ni'], p['B'], N, dtype, dn)
    out['tau_auger'] = tau_auger(p['ni'], N, dtype, dn, p['Cn'], p['Cp'], material if material == 'Si' else None)
    inv = 1 / out['tau_rad'] + 1 / out['tau_auger']
    if tau_n0 is not None:
        out['tau_srh'] = tau_srh(p['ni'], N, dtype, dn, tau_n0, tau_p0 if tau_p0 else tau_n0, Et_minus_Ei)
        inv = inv + 1 / out['tau_srh']
    out['tau_bulk'] = 1 / inv
    if ax is None:
        fig, ax = plt.subplots(figsize=(6.5, 4.4))
    ax.loglog(dn, out['tau_rad'] * 1e6, label=r'$\tau_{rad}$')
    ax.loglog(dn, out['tau_auger'] * 1e6, label=r'$\tau_{Auger}$')
    if tau_n0 is not None: ax.loglog(dn, out['tau_srh'] * 1e6, label=r'$\tau_{SRH}$')
    ax.loglog(dn, out['tau_bulk'] * 1e6, 'k', lw=2.2, label=r'$\tau_{bulk}$')
    ax.set_xlabel(r'Excess Carrier Density $\Delta n$ (cm$^{-3}$)'); ax.set_ylabel('Lifetime (µs)')
    ax.set_title(label or f'{material}, {dtype}-type, N = {N:.1e} cm$^{{-3}}$', fontsize=10); ax.legend(fontsize=8)
    i = np.argmin(abs(dn - 1e15))
    print(f"{material}: at dn = 1e15 cm^-3  tau_rad = {out.tau_rad[i]*1e6:.3g} µs, tau_Auger = {out.tau_auger[i]*1e6:.3g} µs"
          + (f", tau_SRH = {out.tau_srh[i]*1e6:.3g} µs" if tau_n0 is not None else '') + f" -> tau_bulk = {out.tau_bulk[i]*1e6:.3g} µs")
    return out


def led_iqe(A, B, C, d_nm=None, n=None, ax=None, label='ABC model'):
    """Internal quantum efficiency eta = B n^2 / (A n + B n^2 + C n^3) vs carrier density, and vs current density
    J = q d (A n + B n^2 + C n^3) if the active-layer thickness d (nm) is given. (Grundmann §10.10.)"""
    n = np.logspace(16, 20, 300) if n is None else np.asarray(n)
    R = A * n + B * n ** 2 + C * n ** 3; eta = B * n ** 2 / R
    if ax is None:
        fig, ax = plt.subplots(figsize=(6.5, 4.2))
    if d_nm:
        J = q_C * d_nm * 1e-7 * R
        ax.semilogx(J, eta, label=label); ax.set_xlabel(r'Current Density $J$ (A cm$^{-2}$)')
    else:
        ax.semilogx(n, eta, label=label); ax.set_xlabel(r'Carrier Density $n$ (cm$^{-3}$)')
    ax.set_ylabel('Internal Quantum Efficiency'); ax.set_ylim(0, 1); ax.legend(fontsize=8)
    k = np.argmax(eta); print(f"Peak IQE = {eta[k]:.2f} at n = {n[k]:.2e} cm^-3" + (f", J = {q_C*d_nm*1e-7*R[k]:.3g} A/cm^2" if d_nm else ''))
    return n, eta


# ---------------- surface recombination with interface states and fixed charge (extended SRH)
def _surface_potential(Qf, n_b, p_b, eps_r):
    """Surface potential psi_s (V) from charge neutrality Q_f + Q_sc(psi_s) = 0 (Girisch, Mertens & De Keersmaecker,
    IEEE Trans. Electron Devices 35, 203 (1988)); interface-trap charge neglected. psi_s > 0 bends the bands down."""
    eps = eps_r * eps0
    def Qsc(ps):
        u = ps / Vt
        f = p_b * (np.exp(-u) + u - 1) + n_b * (np.exp(u) - u - 1)
        return -np.sign(ps) * np.sqrt(2 * q_C * Vt * eps * max(f, 0))
    g = lambda ps: Qf * q_C + Qsc(ps)
    return brentq(g, -2.0, 2.0)


def surface_recombination(Dit, Qf, N, dtype, dn, sigma_n=1e-15, sigma_p=1e-16, ni=9.65e9, Eg=1.12, eps_r=11.7, verbose=True):
    """Seff (cm/s) and J0s (fA/cm^2) for a surface with uniform interface-state density Dit (eV^-1 cm^-2),
    capture cross sections sigma_n, sigma_p (cm^2) and fixed charge Qf (cm^-2), on a wafer of doping N (cm^-3)
    at injection dn (cm^-3). Extended SRH integral over the gap (Aberle, Glunz & Warta, J. Appl. Phys. 71, 4422 (1992))."""
    n_b, p_b, n0, p0 = carrier_densities(ni, N, dtype, dn)
    psi_s = _surface_potential(Qf, n_b, p_b, eps_r)
    ns, ps = n_b * np.exp(psi_s / Vt), p_b * np.exp(-psi_s / Vt)
    E = np.linspace(-Eg / 2, Eg / 2, 801)                       # relative to midgap
    n1 = ni * np.exp(E / kT); p1 = ni * np.exp(-E / kT)
    integrand = vth * Dit * (ns * ps - ni ** 2) / ((ns + n1) / sigma_p + (ps + p1) / sigma_n)
    Us = np.trapezoid(integrand, E) if hasattr(np, 'trapezoid') else np.trapz(integrand, E)
    Seff = Us / dn
    J0s = q_C * Us * ni ** 2 / (n_b * p_b - ni ** 2) * 1e15   # fA/cm^2
    if verbose:
        print(f"Qf = {Qf:+.1e} cm^-2 -> psi_s = {psi_s:+.3f} V, n_s = {ns:.2e}, p_s = {ps:.2e} cm^-3; "
              f"Seff = {Seff:.3g} cm/s, J0s = {J0s:.3g} fA/cm^2")
    return dict(Seff=Seff, J0s=J0s, psi_s=psi_s, ns=ns, ps=ps)


def seff_vs_qf(Dit, N, dtype, dn=1e15, sigma_n=1e-15, sigma_p=1e-16, ni=9.65e9, Eg=1.12, eps_r=11.7, Qf=None, ax=None, label=None):
    """Sweep the fixed charge and plot Seff and J0s. Returns a DataFrame."""
    Qf = np.concatenate([-np.logspace(13, 10, 60), [0], np.logspace(10, 13, 60)]) if Qf is None else np.asarray(Qf)
    rows = [surface_recombination(Dit, q, N, dtype, dn, sigma_n, sigma_p, ni, Eg, eps_r, verbose=False) for q in Qf]
    df = pd.DataFrame(rows); df.insert(0, 'Qf', Qf)
    if ax is None:
        fig, ax = plt.subplots(figsize=(6.8, 4.4))
    ax.plot(Qf, df.Seff, lw=2, label=label or f'$D_{{it}}$ = {Dit:.0e} eV$^{{-1}}$cm$^{{-2}}$')
    ax.set_xscale('symlog', linthresh=1e10); ax.set_yscale('log')
    ax.set_xlabel(r'Fixed Charge $Q_f$ (cm$^{-2}$)'); ax.set_ylabel(r'$S_{eff}$ (cm s$^{-1}$)')
    ax2 = ax.twinx(); ax2.set_yscale('log'); ax2.plot(Qf, df.J0s, 'C3--', lw=1.2); ax2.set_ylabel(r'$J_{0s}$ (fA cm$^{-2}$)', color='C3')
    ax.set_title(f'{dtype}-type, N = {N:.0e} cm$^{{-3}}$, Δn = {dn:.0e} cm$^{{-3}}$', fontsize=10); ax.legend(fontsize=8, loc='lower center')
    k = np.argmax(df.Seff); print(f"Seff peaks at Qf = {Qf[k]:+.1e} cm^-2 ({df.Seff[k]:.3g} cm/s); "
                                  f"at Qf = {Qf[0]:+.0e}: {df.Seff.iloc[0]:.3g} cm/s; at Qf = {Qf[-1]:+.0e}: {df.Seff.iloc[-1]:.3g} cm/s")
    return df


# ---------------- effective lifetime and implied Voc
def effective_lifetime(material, N, dtype, W_um, S=None, J0s=None, tau_n0=None, tau_p0=None, Et_minus_Ei=0.0, D_cm2s=None,
                       JL_mAcm2=40.0, dn=None, ax=None, label=None, **override):
    """1/tau_eff = 1/tau_rad + 1/tau_Auger + 1/tau_SRH + 1/tau_surface for a wafer of thickness W with two identical
    surfaces given either as S (cm/s; Sproul, J. Appl. Phys. 76, 2851 (1994): tau_s = W/2S + (W/pi)^2/D) or as J0s
    (fA/cm^2; Kane & Swanson 1985: 1/tau_s = 2 J0s (N + dn) / (q ni^2 W)). Prints tau_eff and the implied Voc at the
    1-sun operating point where generation JL/(qW) equals dn/tau_eff."""
    p = dict(RECOMB[material]); p.update(override); ni = p['ni']
    dn = np.logspace(12, 17, 300) if dn is None else np.asarray(dn)
    W = W_um * 1e-4
    inv = 1 / tau_radiative(ni, p['B'], N, dtype, dn) + 1 / tau_auger(ni, N, dtype, dn, p['Cn'], p['Cp'], material if material == 'Si' else None)
    if tau_n0 is not None: inv = inv + 1 / tau_srh(ni, N, dtype, dn, tau_n0, tau_p0 if tau_p0 else tau_n0, Et_minus_Ei)
    tau_bulk = 1 / inv
    if S is not None:
        D = D_cm2s or (27.0 if dtype == 'p' else 12.0)
        tau_s = W / (2 * S) + (W / np.pi) ** 2 / D
        inv = inv + 1 / tau_s
    if J0s is not None:
        inv = inv + 2 * (J0s * 1e-15) * (N + dn) / (q_C * ni ** 2 * W)
    tau_eff = 1 / inv
    surf = (f'S = {S:g} cm/s' if S is not None else '') + (' + ' if S is not None and J0s is not None else '') + (f'$J_{{0s}}$ = {J0s:g} fA/cm$^2$' if J0s is not None else '')
    if ax is None:
        fig, ax = plt.subplots(figsize=(6.5, 4.4))
    if not any(l.get_label() == r'$\tau_{bulk}$' for l in ax.get_lines()):
        ax.loglog(dn, tau_bulk * 1e6, 'k--', lw=1.2, label=r'$\tau_{bulk}$')
    ax.loglog(dn, tau_eff * 1e6, lw=2.2, label=label or (r'$\tau_{eff}$, ' + surf))
    ax.set_xlabel(r'Excess Carrier Density $\Delta n$ (cm$^{-3}$)'); ax.set_ylabel('Lifetime (µs)'); ax.legend(fontsize=8)
    ax.set_title(f'{material}, {dtype}-type {N:.0e} cm$^{{-3}}$, W = {W_um:g} µm', fontsize=10)
    G = JL_mAcm2 * 1e-3 / (q_C * W)
    f = dn / tau_eff - G
    if f[0] < 0 < f[-1]:
        k = np.searchsorted(f, 0); dn_op = dn[k]; tau_op = tau_eff[k]
        n, pp, n0, p0 = carrier_densities(ni, N, dtype, dn_op)
        iVoc = Vt * np.log(n * pp / ni ** 2)
        print(f"[{surf}] 1-sun operating point: dn = {dn_op:.2e} cm^-3, tau_eff = {tau_op*1e6:.3g} µs, implied Voc = {iVoc*1000:.0f} mV")
        ax.axvline(dn_op, color='0.5', ls=':', lw=1)
    return pd.DataFrame({'dn': dn, 'tau_bulk': tau_bulk, 'tau_eff': tau_eff})


def S_from_J0(J0s_fAcm2, N, ni=9.65e9):
    """Low-injection conversion S = J0s N / (q ni^2) for a surface on a wafer of doping N."""
    S = J0s_fAcm2 * 1e-15 * N / (q_C * ni ** 2); print(f"J0s = {J0s_fAcm2} fA/cm^2 on N = {N:.0e} cm^-3 -> S = {S:.3g} cm/s"); return S

print('Module A loaded: bulk_lifetime_plot, tau_radiative/auger/srh, led_iqe, surface_recombination, seff_vs_qf, effective_lifetime, S_from_J0. Presets:', ', '.join(RECOMB))


In [ ]:
# YOUR CODE HERE (Task 1.1) — replace the material, doping and SRH parameters with your own (cite the sources in the report)
fig, ax = plt.subplots(figsize=(6.8, 4.5))
res = bulk_lifetime_plot('Si', N=1.5e16, dtype='p', tau_n0=100e-6, tau_p0=100e-6, Et_minus_Ei=0.0, ax=ax)
# Reported lifetimes from your corpus: (dn, tau in s, label) — add one point per article
reported = [(1e15, 200e-6, 'author2021, QSSPC, passivated wafer')]
for dn_r, tau_r, lab in reported:
    ax.plot(dn_r, tau_r * 1e6, 'o', ms=7, label=lab)
ax.legend(fontsize=7)
fig.savefig('bulk_lifetime.png', dpi=200, bbox_inches='tight')

# Emitters: IQE vs current density from ABC coefficients (A in s^-1, B in cm^3/s, C in cm^6/s; d = active-layer thickness)
# fig, ax = plt.subplots(); led_iqe(A=1e7, B=2e-11, C=2e-30, d_nm=9, ax=ax)


**Task 1.1 Answer** *(10 points)*

**Lifetimes (or ABC/IQE data) reported in your corpus** (value, method, sample, bibtex key):

| value | method | sample | source |
|---|---|---|---|
| ... | ... | ... | ... |

**Parameters used in the calculator and their sources ($n_i$, $B$, $C$ or Richter; doping; SRH estimate):** *(...)*

**Which mechanism limits the bulk, by how much the best reported lifetime falls short of the intrinsic limit, and the operating injection level (or current density):** *(three or four sentences)*


### 1.2 Sources of SRH recombination and bulk passivation (15 points)

Identify, from your corpus, the bulk defects that limit your material — metal impurities (Fe, Cr, Cu in silicon), native point defects and complexes (B–O, $V_{Cd}$, antisites), dislocations and grain boundaries, ion-migration-related defects in perovskites — and the **bulk passivation** strategies used against them: avoiding the defect (feedstock purity, growth control), removing it (phosphorus or aluminum gettering, CdCl$_2$ treatment), or deactivating it (hydrogenation from SiN$_x$:H firing or forming-gas anneal, Se alloying, additive engineering, regeneration treatments). For each strategy, record the *process* (what is deposited or applied, at what temperature, for how long, in what ambient) and the evidence that it worked (lifetime before/after, diffusion length, $V_{oc}$). Note that deep-level identification requires a dedicated measurement — DLTS, lifetime spectroscopy, FTIR, PL spectroscopy — and that many device papers only infer "defects" from a lifetime; say which kind of evidence each paper offers.

**Extend `survey.csv`.** Add the columns below and fill them for your **top-20 rows**; write `not reported` when the article does not give the item (those entries are the gaps your narrative must name). The Task 2 and Task 3–4 columns are listed with their tasks; all are validated together in the cell under Task 2.3.

| column | content |
|---|---|
| `lifetime_value` | the reported lifetime with units and sample (e.g. `1.2 ms, passivated n-Cz wafer`), or IQE/ABC data for emitters |
| `lifetime_method` | one of `QSSPC`, `PCD`, `TRPL`, `PLQY`, `PL imaging`, `EL`, `EQE/JV fit`, `other`, `not reported`, `not applicable` |
| `bulk_limit` | the mechanism or defect the article identifies as limiting the bulk (e.g. `Fe-B`, `B-O`, `grain boundaries`, `Auger at 1e19`, `intrinsic`) |
| `bulk_passivation_process` | the treatment and its conditions (e.g. `P-gettering 850 C 30 min`, `CdCl2 anneal 400 C air`, `FGA 400 C`, `none`) |

**Write the first part of the Passivation subsection** (10 points, report): the bulk mechanisms and defects that matter for your device, the bulk passivation processes used against them with their conditions, and the lifetime levels achieved, with the intrinsic-limit comparison from Task 1.1 as the yardstick. Cite the articles and name the gaps.


**Task 1.2 Answer** *(the columns are checked by the validation cell under Task 2.3; the narrative is graded from the report)*

**Bulk defects and passivation processes in your corpus (one line each: defect — process — evidence — bibtex key):** *(...)*

**Report status checklist:**

* ☐ Bulk mechanisms and limiting defects described with citations
* ☐ Bulk passivation processes described with their conditions
* ☐ Reported lifetimes compared with the intrinsic limit


---
## Task 2 — Surface recombination and surface passivation (25 points)

Completes the **Passivation** subsection. Points: 2.1 — 7 (notebook); 2.2 — 8 (notebook); 2.3 — 10 (5 validation, 5 report; the surface narrative is graded with the Task 1.2 narrative as one subsection).

### Background: surfaces, $S$, $J_{0s}$, $D_{it}$ and $Q_f$

A surface ends the crystal and leaves a continuum of states across the gap, $D_{it}(E)$ in eV$^{-1}$ cm$^{-2}$. Recombination through them is written as a **surface recombination velocity**, $U_s = S\,\Delta n_s$: $10^5$–$10^7$ cm s$^{-1}$ on a bare or metal-covered semiconductor, below 10 cm s$^{-1}$ on well-passivated silicon (Grundmann, Ch. 10; Sproul, J. Appl. Phys. 76, 2851 (1994)). On a doped or charged surface $S$ depends on injection and doping, so the surface is more usefully characterized by its **saturation current density** $J_{0s}$, defined through $U_s = J_{0s}(np - n_i^2)/(q n_i^2)$; in low injection the two are related by $J_{0s} = q n_i^2 S/N$ (Kane & Swanson, Proc. 18th IEEE PVSC, 578 (1985); Cuevas et al., Sol. Energy Mater. Sol. Cells 184, 38 (2018)). Two routes reduce $U_s$ (Aberle, Glunz & Warta, J. Appl. Phys. 71, 4422 (1992); Dingemans & Kessels, J. Vac. Sci. Technol. A 30, 040802 (2012)): **chemical passivation** lowers $D_{it}$ by completing the bonds (a native or grown oxide, hydrogen, a lattice-matched epitaxial layer), and **field-effect passivation** uses fixed charge $Q_f$ in the film, or doping under the surface, to remove one carrier type from the surface so that the recombination rate — which needs both — collapses. The polarity has to match the surface: negative charge (AlO$_x$) suits $p$ and $p^+$ surfaces, positive charge (SiN$_x$:H) suits $n$ and $n^+$; the wrong polarity drives the surface toward $n_s \approx p_s$, where $U_s$ peaks.

### 2.1 What your corpus reports about surfaces (7 points)

Collect the surface and interface recombination data for your device: $S$ or $S_{eff}$, $J_{0s}$ (or $J_{0e}$ for diffused surfaces), $D_{it}$ and $Q_f$ where measured (C–V, conductance, corona charging, Kelvin probe), TRPL-derived surface velocities for thin films, sidewall velocities for micro-LEDs, and the passivation stack and process each refers to. Convert between $S$ and $J_{0s}$ with `S_from_J0` where only one is given (state the doping you assumed). Give the range, the best values, and what sample structure they were measured on. For III–V emitters report the surface/interface velocity of the relevant heterointerface (window, cladding, sidewall); for perovskites and organics report what the papers call interface recombination velocity or the surface contribution to the TRPL lifetime, and say how it was separated from the bulk.


**Task 2.1 Answer** *(7 points)*

| quantity | value(s) | passivation stack and process | sample / method | source |
|---|---|---|---|---|
| $S$ or $S_{eff}$ (cm/s) | ... | ... | ... | ... |
| $J_{0s}$ or $J_{0e}$ (fA/cm²) | ... | ... | ... | ... |
| $D_{it}$ (eV⁻¹ cm⁻²) | ... | ... | ... | ... |
| $Q_f$ (cm⁻²) | ... | ... | ... | ... |

**Best reported surface and the structure it was measured on; what is not reported in your field:** *(two or three sentences)*


### 2.2 Calculators: $S_{eff}$ vs. fixed charge, and the effective lifetime (8 points)

**(a) Fixed charge.** `seff_vs_qf` solves the extended SRH model for a surface with a uniform $D_{it}$, capture cross sections $\sigma_n$, $\sigma_p$ and a fixed charge $Q_f$: it finds the surface potential from charge neutrality, the surface carrier densities, and integrates the recombination rate over the gap. Run it for the $D_{it}$ you selected in Task 2.1 (or a representative value if none is reported), the doping of your absorber and the operating injection level, and then once more for the opposite doping type. Identify in the plot: the $Q_f$ at which $S_{eff}$ peaks and why it sits where it does; the charge polarity and magnitude that passivate *your* surface; and the $D_{it}$ needed to reach the $S$ or $J_{0s}$ your corpus reports. (Model limits: uniform $D_{it}$, single $\sigma$ pair, no interface-trap charge, no surface doping — a diffused emitter needs the doping profile instead, which this model does not include.)

**(b) Effective lifetime and implied $V_{oc}$.** `effective_lifetime` combines the bulk terms of Task 1 with two identical surfaces given as $S$ (Sproul: $\tau_s = W/2S + (W/\pi)^2/D$) or as $J_{0s}$ (Kane–Swanson: $1/\tau_s = 2J_{0s}(N + \Delta n)/(q n_i^2 W)$), plots $\tau_{eff}(\Delta n)$ and finds the one-sun operating point where generation balances recombination, with the implied open-circuit voltage $iV_{oc} = (kT/q)\ln[(n_0 + \Delta n)(p_0 + \Delta n)/n_i^2]$. Run it with the bulk parameters from Task 1 and the best and typical surfaces from Task 2.1, and report what limits $\tau_{eff}$ at the operating point and how much $V_{oc}$ the best surface buys over the typical one. Emitters: use the sidewall form $A = A_0 + 4S/d$ (Wong et al., Opt. Express 26, 21324 (2018)) in `led_iqe` to show how IQE falls as the mesa size $d$ shrinks for the sidewall $S$ you found.


In [ ]:
# YOUR CODE HERE (Task 2.2a) — Seff and J0s vs fixed charge for your Dit, doping and injection level
fig, ax = plt.subplots(figsize=(7, 4.5))
df_q = seff_vs_qf(Dit=1e11, N=1.5e16, dtype='p', dn=1e15, sigma_n=1e-15, sigma_p=1e-16, ax=ax)
fig.savefig('seff_vs_qf.png', dpi=200, bbox_inches='tight')

# single points, e.g. the charge of your passivation film:
surface_recombination(Dit=1e11, Qf=-3e12, N=1.5e16, dtype='p', dn=1e15)     # AlOx-like negative charge
surface_recombination(Dit=1e11, Qf=+2e12, N=1.5e16, dtype='p', dn=1e15)     # SiNx-like positive charge


In [ ]:
# YOUR CODE HERE (Task 2.2b) — effective lifetime and implied Voc with your bulk and surface values
fig, ax = plt.subplots(figsize=(6.8, 4.5))
res_typ = effective_lifetime('Si', N=1.5e16, dtype='p', W_um=150, S=100, tau_n0=1e-3, ax=ax)       # typical surface
res_best = effective_lifetime('Si', N=1.5e16, dtype='p', W_um=150, J0s=5, tau_n0=1e-3, ax=ax)     # best surface, as J0s
fig.savefig('effective_lifetime.png', dpi=200, bbox_inches='tight')

# Emitters (micro-LED sidewalls): A = A0 + 4*S/d
# for d_um in (100, 20, 5):
#     A = 1e7 + 4 * 1e4 / (d_um * 1e-4); led_iqe(A, 2e-11, 2e-30, d_nm=9, ax=ax, label=f'd = {d_um} µm')


**Task 2.2 Answer** *(8 points)*

**Where $S_{eff}$ peaks and why; the polarity and magnitude of $Q_f$ that passivate your surface; the $D_{it}$ needed to reach the reported $S$ or $J_{0s}$:** *(...)*

**At the one-sun operating point: what limits $\tau_{eff}$, and the $iV_{oc}$ gain of the best surface over the typical one (or the IQE loss with mesa size for emitters):** *(...)*


### 2.3 Surface passivation processes and the Passivation narrative (10 points)

Add the Task 2 columns to `survey.csv` for your top-20 rows and run the validation cell (which also checks the Task 1 and Task 3–4 columns, so complete those before the final run):

| column | content |
|---|---|
| `surface_passivation_stack` | the film(s) or layer(s) on each surface, with thicknesses (e.g. `AlOx 10 nm / SiNx 75 nm (front), AlOx/SiNx (rear)`; `AlGaAs window`; `Al2O3 ALD on sidewalls`) |
| `surface_passivation_process` | deposition and post-deposition steps with conditions (e.g. `ALD TMA/H2O 200 C + firing 800 C`; `PECVD 400 C + FGA`; `MOCVD in situ`) |
| `S_or_J0s` | the reported value with units and which surface (e.g. `J0s = 3 fA/cm2 (rear)`; `S = 1e4 cm/s (sidewall)`), or `not reported` |
| `Dit_or_Qf` | reported interface-state density and/or fixed charge, or `not reported` |

**Write the second part of the Passivation subsection** (5 points, report) from the table: the passivation materials and processes used in your field, the interface structure and chemistry they produce (the interfacial oxide, hydrogen, the epitaxial interface), the $D_{it}$, $Q_f$ and $S$ or $J_{0s}$ they reach, and the effect on the device — $V_{oc}$ or dark current, IQE or EQE — with the calculator results of Task 2.2 as the quantitative link. Name what your corpus does not report.


In [ ]:
# ============================================================
# HW03 MODULE D — survey.csv extension and validation (instructor-provided; run, do not modify)
# ============================================================
import pandas as pd, numpy as np

HW1_COLUMNS = ["first_author", "year", "title", "doi", "device_type", "absorber_emitter",
               "key_fom_name", "key_fom_value", "processing_disclosed", "top20", "bibtex_key"]
PASS_COLUMNS = ["lifetime_value", "lifetime_method", "bulk_limit", "bulk_passivation_process",
                "surface_passivation_stack", "surface_passivation_process", "S_or_J0s", "Dit_or_Qf"]
OPT_COLUMNS = ["front_texture", "texture_process", "arc_stack", "arc_process", "rear_reflector_stack",
               "other_optical", "optical_metric"]
NEW_COLUMNS = PASS_COLUMNS + OPT_COLUMNS
LIFETIME_METHODS = {"qsspc", "pcd", "trpl", "plqy", "pl imaging", "el", "eqe/jv fit", "other", "not reported", "not applicable"}


def validate_survey(filename="survey.csv"):
    df = pd.read_csv(filename)
    missing = [c for c in HW1_COLUMNS + NEW_COLUMNS if c not in df.columns]
    assert not missing, f"Missing columns: {missing}"
    top = df[df["top20"].astype(str).str.strip().str.lower() == "yes"].copy()
    assert len(top) == 20, f"Exactly 20 rows must have top20 == yes (found {len(top)})."
    for c in NEW_COLUMNS:
        top[c] = top[c].astype(str).str.strip()
    empties = {c: int((top[c].isin(["", "nan", "NaN"])).sum()) for c in NEW_COLUMNS}
    empties = {k: v for k, v in empties.items() if v}
    assert not empties, f"Empty cells in top-20 rows (write 'not reported' if the article does not give it): {empties}"
    bad = sorted(set(top["lifetime_method"].str.lower()) - LIFETIME_METHODS)
    assert not bad, f"Unrecognized lifetime_method {bad}; use one of {sorted(LIFETIME_METHODS)}"
    nr = {c: int((top[c].str.lower() == "not reported").sum()) for c in NEW_COLUMNS}
    print(f"Corpus: {len(df)} articles; top-20 rows checked: {len(top)}")
    print("'not reported' counts among the top 20 (the gaps your narrative must name):")
    for c, v in nr.items():
        print(f"   {c:28s} {v:2d} / 20")
    print("\nValidation PASSED. The summaries below are part of your submission:")
    with pd.option_context("display.max_rows", 50, "display.width", 120):
        print("\nLifetime measurement methods (top 20):"); print(top["lifetime_method"].str.lower().value_counts().to_string())
        print("\nSurface passivation stacks (top 20):"); print(top["surface_passivation_stack"].value_counts().to_string())
        print("\nFront texture (top 20):"); print(top["front_texture"].value_counts().to_string())
        print("\nARC stacks (top 20):"); print(top["arc_stack"].value_counts().to_string())
        print("\nRear reflector stacks (top 20):"); print(top["rear_reflector_stack"].value_counts().to_string())
    return df

print("Module D loaded: validate_survey(). New columns required for the top-20 rows:", ", ".join(NEW_COLUMNS))


In [ ]:
# Tasks 1–4 validation — upload the extended survey.csv to this session, then run this cell (re-run after adding the Task 3–4 columns).
survey = validate_survey('survey.csv')


**Task 2.3 Answer** *(5 points for a passing validation cell with the summary printed above; the narrative is graded from the report)*

**Items most often 'not reported' for passivation in your corpus, and the articles that do report them:** *(...)*

**Report status checklist:**

* ☐ Surface passivation materials and processes described with conditions
* ☐ Interface structure and chemistry described
* ☐ $D_{it}$, $Q_f$, $S$ or $J_{0s}$ values given with citations and linked to device performance
* ☐ Gaps in the corpus named explicitly


---
## Task 3 — Optical surfaces: reflectance, escape cone and texture (20 points)

Begins the **Optical Surfaces and Coatings** subsection. Points: 3.1 — 8 (notebook); 3.2 — 12 (4 notebook, 8 report).

### Background

At a planar interface between media of complex index $N = n + i\kappa$ the normal-incidence reflectance is $R = |(N_1 - N_2)/(N_1 + N_2)|^2$ (Fresnel; Macleod, *Thin-Film Optical Filters*, 3rd ed. (2001), Ch. 2), which for a bare semiconductor in air is 18–40 % — the first loss every absorber must deal with, and the reason every solar cell, photodiode and LED carries a coating or a texture or both. For an emitter the problem is inverted: light generated inside a medium of index $n$ can only leave through the escape cone $\theta < \theta_c = \arcsin(1/n)$, a fraction $1/(4n^2)$ of the isotropic emission per pass (Yablonovitch, J. Opt. Soc. Am. 72, 899 (1982)); the same $4n^2$ is the limit on path-length enhancement that a randomizing texture gives an absorber. Texture, encapsulant, die shaping and roughening all work by changing the index step the light sees or by randomizing its direction.

### 3.1 Bare-surface reflectance and the escape cone (8 points)

Load the $n,\kappa$ of your absorber or emitter (the Homework 2 loader is included below; use the same data set, or your own CSV) and compute $R(\lambda)$ of the planar, uncoated surface in air. Report the weighted average that matters for your device: AM1.5G photon-flux-weighted $\langle R \rangle$ over the absorption range for a solar cell; $R$ at the detection wavelength for a photodetector; for an emitter, run `escape_cone` for your material against air and against an encapsulant ($n \approx 1.5$) and report the critical angle and single-pass extraction fraction. Then repeat the solar-cell calculation with the cell under an encapsulant ($n_0 = 1.5$) instead of air, which is how a module sees it.


In [ ]:
# ============================================================
# HW02 MODULE B — optical constants and absorption (instructor-provided; run, do not modify)
# Loads n,k from the refractiveindex.info database (Polyanskiy, Sci. Data 11, 94 (2024); CC0 public domain),
# computes alpha = 4*pi*k/lambda, and provides absorption-depth, Tauc and emitter-overlay helpers.
# ============================================================
import io, urllib.request
import numpy as np, pandas as pd, yaml
import matplotlib.pyplot as plt
from scipy.optimize import brentq

RII_BASE = 'https://raw.githubusercontent.com/polyanskiy/refractiveindex.info-database/master/database/data/'
HC_EV_NM = 1239.84193   # h*c in eV*nm

# Database paths verified September 2026. To use another data set, open the material page on
# refractiveindex.info, click "Data [YAML]" and paste the path after '.../database/data/'.
NK_SOURCES = {
    'Si':        ('main/Si/nk/Green-2008.yml',          'Green, Sol. Energy Mater. Sol. Cells 92, 1305 (2008); 0.25–1.45 µm'),
    'Ge':        ('main/Ge/nk/Nunley.yml',              'Nunley et al., J. Vac. Sci. Technol. B 34, 061205 (2016)'),
    'GaAs':      ('main/GaAs/nk/Papatryfonos.yml',      'Papatryfonos et al., AIP Adv. 11, 025327 (2021); 0.26–1.88 µm'),
    'InP':       ('main/InP/nk/Aspnes.yml',             'Aspnes & Studna, Phys. Rev. B 27, 985 (1983); 0.21–0.83 µm'),
    'GaP':       ('main/GaP/nk/Aspnes.yml',             'Aspnes & Studna 1983'),
    'InAs':      ('main/InAs/nk/Adachi.yml',            'Adachi, J. Appl. Phys. 66, 6030 (1989) (model); 0.21–12.4 µm'),
    'GaSb':      ('main/GaSb/nk/Adachi.yml',            'Adachi 1989 (model); 0.21–12.4 µm'),
    'InSb':      ('main/InSb/nk/Adachi.yml',            'Adachi 1989 (model)'),
    'GaN':       ('main/GaN/nk/Kawashima.yml',          'Kawashima et al., J. Appl. Phys. 82, 3528 (1997); thin film'),
    'AlN':       ('main/AlN/nk/Beliaev1.yml',           'Beliaev et al. 2021; sputtered thin film (check stoichiometry note on the site)'),
    'CdTe':      ('main/CdTe/nk/Treharne.yml',          'Treharne et al., J. Phys.: Conf. Ser. 286, 012038 (2011); 0.3–1.5 µm'),
    'CdS':       ('main/CdS/nk/Treharne.yml',           'Treharne et al. 2011'),
    'ZnSe':      ('main/ZnSe/nk/Adachi.yml',            'Adachi (model)'),
    'AlGaAs (x=0.22)': ('other/semiconductor alloys/AlAs-GaAs/nk/Papatryfonos-21.9.yml', 'Papatryfonos et al. 2021; other compositions on the site'),
    'AlGaAs (x=0.41)': ('other/semiconductor alloys/AlAs-GaAs/nk/Papatryfonos-41.1.yml', 'Papatryfonos et al. 2021'),
    'GaInP':     ('other/semiconductor alloys/GaP-InP/nk/Schubert.yml', 'Schubert et al., J. Appl. Phys. 77, 3416 (1995); Ga0.5In0.5P'),
    'InGaAs':    ('other/semiconductor alloys/GaAs-InAs/nk/Adachi.yml', 'Adachi 1989 (model); In0.53Ga0.47As'),
    'MAPbI3':    ('other/perovskite/CH3NH3PbI3/nk/Phillips.yml', 'Phillips et al., Sol. Energy Mater. Sol. Cells 147, 327 (2016)'),
    'CsPbBr3':   ('other/perovskite/CsPbBr3/nk/Brennan.yml',     'Brennan et al. 2019'),
    'CsPbI3':    ('other/perovskite/CsPbI3/nk/Aftenieva.yml',    'Aftenieva et al.'),
    'P3HT:PCBM': ('other/mixed organic/P3HT-PC61BM/nk/Stelling.yml', 'Stelling et al., Sci. Rep. 7, 42530 (2017)'),
    'PTB7':      ('organic/(C41H53FO4S4)n - PTB7/nk/Kamptner-o.yml', 'Kamptner et al. (ordinary index)'),
    'ITO':       ('other/mixed crystals/In2O3-SnO2/nk/Konig.yml',  'König et al., ACS Nano 8, 6182 (2014)'),
    'AZO':       ('other/doped crystals/Al-ZnO/nk/Treharne.yml',   'Treharne et al. 2011'),
    'TiO2 (film)': ('main/TiO2/nk/Sarkar.yml',          'Sarkar et al. 2019; thin film'),
    'MoO3':      ('main/MoO3/nk/Vos.yml',               'Vos et al.'),
}
NOT_IN_DATABASE = ('InGaN, AlGaN, CIGS (CuIn1-xGaxSe2), a-Si:H, HgCdTe and most alloys are NOT in refractiveindex.info. '
                   'Take n,k from a paper in your corpus (ellipsometry data are often in the SI) or from the PV Lighthouse '
                   'refractive-index library (free account), save two or three columns (wavelength_nm, n, k) as a CSV, '
                   'and load it with load_nk_csv().')


def _fetch(path):
    return urllib.request.urlopen(RII_BASE + urllib.parse.quote(path), timeout=60).read().decode()

import urllib.parse

def load_nk(material_or_path):
    """Return a DataFrame [wavelength_nm, n, k] from refractiveindex.info.
    Accepts a key of NK_SOURCES or a database path such as 'main/Si/nk/Green-2008.yml'."""
    path, label = NK_SOURCES.get(material_or_path, (material_or_path, material_or_path))
    doc = yaml.safe_load(_fetch(path))
    frames = []
    for block in doc['DATA']:
        typ = block.get('type', '')
        if typ.startswith('tabulated'):
            arr = np.loadtxt(io.StringIO(block['data']))
            arr = np.atleast_2d(arr)
            df = pd.DataFrame({'wavelength_nm': arr[:, 0] * 1000.0})
            if typ == 'tabulated nk':
                df['n'], df['k'] = arr[:, 1], arr[:, 2]
            elif typ == 'tabulated n':
                df['n'] = arr[:, 1]
            elif typ == 'tabulated k':
                df['k'] = arr[:, 1]
            frames.append(df)
        else:
            print(f"Note: '{path}' stores {typ} (a dispersion formula for n, no k). alpha cannot be computed from it.")
    if not frames:
        raise ValueError('No tabulated data in this file.')
    df = frames[0]
    for other in frames[1:]:      # merge separate n and k tables on wavelength by interpolation
        for col in ('n', 'k'):
            if col in other and col not in df:
                df[col] = np.interp(df['wavelength_nm'], other['wavelength_nm'], other[col])
    if 'k' not in df: df['k'] = np.nan
    if 'n' not in df: df['n'] = np.nan
    df = df.sort_values('wavelength_nm').reset_index(drop=True)
    df.attrs['source'] = label; df.attrs['path'] = path
    df.attrs['reference'] = doc.get('REFERENCES', '').strip()
    print(f"Loaded {path}: {len(df)} points, {df.wavelength_nm.min():.0f}–{df.wavelength_nm.max():.0f} nm. "
          f"Reference: {df.attrs['reference'][:120]}")
    return df


def load_nk_csv(filename, wavelength_unit='nm'):
    """Load your own n,k data (columns: wavelength, n, k) from a CSV; wavelength_unit 'nm' or 'um'."""
    df = pd.read_csv(filename)
    df.columns = [c.strip().lower() for c in df.columns]
    wcol = [c for c in df.columns if c.startswith('wave') or c in ('lambda', 'lam', 'wl')][0]
    out = pd.DataFrame({'wavelength_nm': df[wcol] * (1000.0 if wavelength_unit == 'um' else 1.0), 'n': df['n'], 'k': df['k']})
    out.attrs['source'] = filename
    return out.sort_values('wavelength_nm').reset_index(drop=True)


def add_alpha(df):
    """Add photon energy (eV) and absorption coefficient alpha (cm^-1) = 4*pi*k/lambda."""
    df = df.copy()
    df['E_eV'] = HC_EV_NM / df['wavelength_nm']
    df['alpha_cm'] = 4 * np.pi * df['k'] / (df['wavelength_nm'] * 1e-7)
    return df


def plot_absorption(df, Eg=None, label=None, lam_marks=(), ax=None):
    """alpha vs wavelength (left) and vs photon energy (right), log scale; optional Eg line and wavelength marks."""
    df = add_alpha(df); d = df[df.alpha_cm > 0]
    fig, axs = plt.subplots(1, 2, figsize=(11, 4.2))
    axs[0].semilogy(d.wavelength_nm, d.alpha_cm, lw=1.8, label=label or df.attrs.get('source', ''))
    axs[0].set_xlabel('Wavelength (nm)'); axs[0].set_ylabel(r'Absorption Coefficient $\alpha$ (cm$^{-1}$)')
    axs[1].semilogy(d.E_eV, d.alpha_cm, lw=1.8)
    axs[1].set_xlabel('Photon Energy (eV)'); axs[1].set_ylabel(r'Absorption Coefficient $\alpha$ (cm$^{-1}$)')
    if Eg:
        axs[1].axvline(Eg, color='k', ls='--', lw=1); axs[1].text(Eg, d.alpha_cm.max(), f' $E_g$ = {Eg} eV', va='top', fontsize=9)
        axs[0].axvline(HC_EV_NM / Eg, color='k', ls='--', lw=1)
    for lam in lam_marks:
        axs[0].axvline(lam, color='C1', ls=':', lw=1.2); axs[0].text(lam, d.alpha_cm.min() * 2, f' {lam:.0f} nm', color='C1', fontsize=8, rotation=90)
    for ax in axs: ax.grid(True, which='major', alpha=0.25)
    axs[0].legend(fontsize=8); plt.tight_layout()
    return fig


def absorption_depth(df, wavelength_nm):
    """1/alpha (µm) at a wavelength, and the single-pass absorbed fraction 1-exp(-alpha*t) for a few thicknesses."""
    df = add_alpha(df)
    a = np.interp(wavelength_nm, df.wavelength_nm, df.alpha_cm)
    if not np.isfinite(a) or a <= 0:
        print(f'alpha is not available at {wavelength_nm} nm (outside the data range or k = 0).'); return np.nan
    depth_um = 1e4 / a
    print(f"lambda = {wavelength_nm:.0f} nm: alpha = {a:.3e} cm^-1, absorption depth 1/alpha = {depth_um:.3g} µm")
    for t_um in (0.1, 0.5, 1, 2, 5, 10, 50, 150):
        print(f"   thickness {t_um:>5} µm -> single-pass absorbed fraction 1-exp(-alpha t) = {1-np.exp(-a*t_um*1e-4):.3f}")
    return depth_um


def tauc_plot(df, kind='direct', fit_range=None, ax=None):
    """Tauc plot (Tauc, Grigorovici & Vancu, Phys. Status Solidi 15, 627 (1966)): (alpha*E)^(1/r) vs E with
    r = 1/2 for allowed direct transitions and r = 2 for allowed indirect. A straight-line fit over fit_range=(E1, E2)
    extrapolated to zero gives an optical-gap estimate. Returns Eg_fit (eV)."""
    df = add_alpha(df); d = df[(df.alpha_cm > 0)].sort_values('E_eV')
    r = 0.5 if kind == 'direct' else 2.0
    y = (d.alpha_cm * d.E_eV) ** (1.0 / r); x = d.E_eV.values
    if ax is None:
        fig, ax = plt.subplots(figsize=(5.5, 4.2))
    ax.plot(x, y, lw=1.6, label=f'({chr(945)}E)^{1/r:g}')
    Eg_fit = np.nan
    if fit_range:
        m = (x >= fit_range[0]) & (x <= fit_range[1])
        if m.sum() >= 3:
            p = np.polyfit(x[m], y[m], 1); Eg_fit = -p[1] / p[0]
            xx = np.linspace(Eg_fit, x[m].max(), 50)
            ax.plot(xx, np.polyval(p, xx), 'k--', lw=1, label=f'linear fit -> $E_g$ = {Eg_fit:.2f} eV')
    ax.set_xlabel('Photon Energy (eV)'); ax.set_ylabel(f'(αE)^{1/r:g}')
    if fit_range:            # show the region around the edge, not the whole spectrum
        lo, hi = fit_range[0] - 0.5, fit_range[1] + 0.3
        ax.set_xlim(lo, hi); w = (x >= lo) & (x <= hi)
        if w.any(): ax.set_ylim(0, np.nanmax(y[w]) * 1.05)
    else:
        ax.set_ylim(0, np.nanpercentile(y, 98) * 1.05)
    ax.legend(fontsize=8); ax.grid(alpha=0.25)
    ax.set_title(f'Tauc Plot, {kind.capitalize()} Transition', fontsize=10)
    return Eg_fit


def overlay_emission(ax, peak_nm, fwhm_nm, label='emission (datasheet)'):
    """Shade the emission band (peak ± FWHM/2) on a wavelength-axis plot — for LED topics."""
    ax.axvspan(peak_nm - fwhm_nm / 2, peak_nm + fwhm_nm / 2, color='C3', alpha=0.15, label=label)
    ax.axvline(peak_nm, color='C3', lw=1.2)
    ax.legend(fontsize=8)


# --- alloy band gaps at 300 K: Eg(x) = (1-x) Eg_A + x Eg_B - b x (1-x).
# Values: Vurgaftman, Meyer & Ram-Mohan, J. Appl. Phys. 89, 5815 (2001); Vurgaftman & Meyer, J. Appl. Phys. 94,
# 3675 (2003) for nitrides. Replace with the values from your own sources and cite them.
ALLOY_GAPS = {
    'InxGa1-xN':   dict(EgA=3.42, EgB=0.78, b=1.4,  A='GaN', B='InN'),
    'AlxGa1-xN':   dict(EgA=3.42, EgB=6.13, b=0.7,  A='GaN', B='AlN'),
    'AlxGa1-xAs':  dict(EgA=1.424, EgB=3.03, b=0.37, A='GaAs', B='AlAs (Gamma valley; direct only for x < ~0.45)'),
    '(AlxGa1-x)0.5In0.5P': dict(EgA=1.91, EgB=2.52, b=0.18, A='Ga0.5In0.5P', B='Al0.5In0.5P (Gamma; direct only for x < ~0.53)'),
    'InxGa1-xAs':  dict(EgA=1.424, EgB=0.354, b=0.477, A='GaAs', B='InAs'),
    'Hg1-xCdxTe':  dict(EgA=-0.15, EgB=1.50, b=0.0, A='HgTe (semimetal)', B='CdTe'),
}

def alloy_gap(alloy, x):
    p = ALLOY_GAPS[alloy]
    return (1 - x) * p['EgA'] + x * p['EgB'] - p['b'] * x * (1 - x)

def composition_for_wavelength(alloy, peak_nm):
    """Solve Eg(x) = hc/lambda for the alloy fraction x that emits at peak_nm (300 K, band-edge emission)."""
    E = HC_EV_NM / peak_nm
    f = lambda x: alloy_gap(alloy, x) - E
    xs = np.linspace(0, 1, 2001); fx = f(xs)
    roots = [brentq(f, xs[i], xs[i + 1]) for i in range(len(xs) - 1) if fx[i] * fx[i + 1] < 0]
    if not roots:
        print(f'{alloy}: no composition gives {peak_nm} nm ({E:.2f} eV) with these parameters.'); return np.nan
    x = roots[0]
    print(f"{alloy}: peak {peak_nm:.0f} nm = {E:.3f} eV  ->  x = {x:.3f}  (Eg({x:.3f}) = {alloy_gap(alloy, x):.3f} eV; "
          f"assumes band-edge emission at 300 K; quantum confinement and strain shift this)")
    return x

print('Module B loaded: load_nk, load_nk_csv, add_alpha, plot_absorption, absorption_depth, tauc_plot, overlay_emission, alloy_gap, composition_for_wavelength')
print('Materials with verified database paths:', ', '.join(NK_SOURCES))
print(NOT_IN_DATABASE)


In [ ]:
# ============================================================
# HW03 MODULE C — thin-film optics: Fresnel reflectance, transfer-matrix method, ARC design, reflectors
# (instructor-provided; run, do not modify). Uses load_nk() from Module B.
# Formalism: H. A. Macleod, Thin-Film Optical Filters, 3rd ed. (IoP, 2001), Ch. 2 (characteristic matrix).
# ============================================================
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# Coating and mirror materials (refractiveindex.info paths verified October 2026). Add your own with load_nk(path).
COATING_SOURCES = {
    'SiNx (PECVD, n~2.1)': ('main/Si3N4/nk/Vogt-2.yml', 'Vogt 2015 (PECVD SiNx, solar-cell film)'),
    'SiNx (PECVD, n~1.9)': ('main/Si3N4/nk/Vogt-1.yml', 'Vogt 2015'),
    'SiO2':   ('main/SiO2/nk/Rodriguez-de Marcos.yml', 'Rodríguez-de Marcos et al. 2016'),
    'Al2O3':  ('main/Al2O3/nk/Franta.yml', 'Franta et al. 2015 (thin film)'),
    'MgF2':   ('main/MgF2/nk/Rodriguez-de Marcos.yml', 'Rodríguez-de Marcos et al. 2017'),
    'TiO2 (film)': ('main/TiO2/nk/Sarkar.yml', 'Sarkar et al. 2019'),
    'Ta2O5':  ('main/Ta2O5/nk/Gao.yml', 'Gao et al. 2012'),
    'ZnS':    ('main/ZnS/nk/Querry.yml', 'Querry 1987'),
    'ITO':    ('other/mixed crystals/In2O3-SnO2/nk/Konig.yml', 'König et al. 2014'),
    'Ag':     ('main/Ag/nk/Johnson.yml', 'Johnson & Christy 1972'),
    'Al':     ('main/Al/nk/McPeak.yml', 'McPeak et al. 2015'),
    'Au':     ('main/Au/nk/Johnson.yml', 'Johnson & Christy 1972'),
}
_cache = {}

def nk_of(material, lam_nm):
    """Complex index N = n + ik interpolated at wavelengths lam_nm. `material` may be a key of COATING_SOURCES or
    NK_SOURCES (Module B), a database path, a DataFrame from load_nk/load_nk_csv, or a number (constant n, k = 0)."""
    if isinstance(material, (int, float, complex)):
        return np.full_like(np.asarray(lam_nm, float), complex(material), dtype=complex)
    if isinstance(material, pd.DataFrame):
        df = material
    else:
        key = COATING_SOURCES.get(material, (material, None))[0]
        if key not in _cache: _cache[key] = load_nk(key)
        df = _cache[key]
    lam = np.asarray(lam_nm, float)
    n = np.interp(lam, df.wavelength_nm, df.n); k = np.interp(lam, df.wavelength_nm, df.k.fillna(0.0))
    out = n + 1j * k
    if lam.min() < df.wavelength_nm.min() or lam.max() > df.wavelength_nm.max():
        print(f"  note: {material if isinstance(material, str) else 'data'} extrapolated outside {df.wavelength_nm.min():.0f}–{df.wavelength_nm.max():.0f} nm (end values held)")
    return out


def fresnel_normal(N1, N2):
    """Normal-incidence power reflectance between media with complex indices N1 (incident) and N2."""
    return np.abs((N1 - N2) / (N1 + N2)) ** 2


def tmm(lam_nm, n_inc, layers, n_sub, theta0_deg=0.0, pol='avg'):
    """Transfer-matrix reflectance/transmittance of a stack. layers = [(material, thickness_nm), ...] from the
    incident medium toward the substrate (both semi-infinite). pol = 's', 'p' or 'avg'. Returns a DataFrame with columns R, Trans, Abs,
    where Abs is the absorptance inside the layers (1 - R - T)."""
    lam = np.asarray(lam_nm, float)
    # Macleod's convention is N = n - ik with phase thickness delta = 2*pi*N*d*cos(theta)/lambda
    N0 = np.conj(nk_of(n_inc, lam)); Ns = np.conj(nk_of(n_sub, lam))
    Nl = [np.conj(nk_of(m, lam)) for m, d in layers]; dl = [d for m, d in layers]
    th0 = np.deg2rad(theta0_deg)
    def run(p):
        # tilted admittances (free-space units); cos(theta_j) from Snell with complex N
        def cosj(N): return np.sqrt(1 - (N0 * np.sin(th0) / N) ** 2 + 0j)
        def eta(N): c = cosj(N); return N * c if p == 's' else N / c
        eta0, etas = eta(N0), eta(Ns)
        B = np.ones_like(lam, dtype=complex); C = etas.copy()
        for N, d in reversed(list(zip(Nl, dl))):
            e = eta(N); delta = 2 * np.pi * N * cosj(N) * d / lam
            cd, sd = np.cos(delta), np.sin(delta)
            B, C = cd * B + 1j * sd / e * C, 1j * e * sd * B + cd * C
        r = (eta0 * B - C) / (eta0 * B + C)
        R = np.abs(r) ** 2
        T = 4 * eta0.real * etas.real / np.abs(eta0 * B + C) ** 2
        return R, T
    if pol == 'avg':
        Rs, Ts = run('s'); Rp, Tp = run('p'); R, T = 0.5 * (Rs + Rp), 0.5 * (Ts + Tp)
    else:
        R, T = run(pol)
    return pd.DataFrame({'wavelength_nm': lam, 'R': R.real, 'Trans': T.real, 'Abs': 1 - R.real - T.real})


# ---------------- spectral weights
def am15g_photon_weight(lam_nm):
    """ASTM G173 AM1.5G photon flux (relative) at the given wavelengths (pvlib)."""
    import pvlib
    spec = pvlib.spectrum.get_reference_spectra()
    lam = np.asarray(lam_nm, float)
    E = np.interp(lam, spec.index.values, spec['global'].values)      # W m^-2 nm^-1
    return E * lam                                                     # photon flux ∝ E * lambda

def emission_weight(lam_nm, peak_nm, fwhm_nm):
    lam = np.asarray(lam_nm, float); s = fwhm_nm / 2.3548
    return np.exp(-0.5 * ((lam - peak_nm) / s) ** 2)

def weighted_average(lam_nm, y, weight):
    """Weighted mean of y(lambda) with weight(lambda) (an array, or 'am15g')."""
    w = am15g_photon_weight(lam_nm) if isinstance(weight, str) else np.asarray(weight)
    return float(np.trapezoid(y * w, lam_nm) / np.trapezoid(w, lam_nm)) if hasattr(np, 'trapezoid') else float(np.trapz(y * w, lam_nm) / np.trapz(w, lam_nm))


def bare_surface_reflectance(material, lam_nm, n_inc=1.0, weight='am15g', label=None, ax=None):
    """R(lambda) of a planar, uncoated surface at normal incidence and its weighted average."""
    lam = np.asarray(lam_nm, float)
    R = fresnel_normal(nk_of(n_inc, lam), nk_of(material, lam))
    Rw = weighted_average(lam, R, weight)
    if ax is None:
        fig, ax = plt.subplots(figsize=(6.8, 4.2))
    ax.plot(lam, 100 * R, lw=2, label=f"{label or material}, bare: ⟨R⟩ = {100*Rw:.1f} %")
    ax.set_xlabel('Wavelength (nm)'); ax.set_ylabel('Reflectance (%)'); ax.set_ylim(0, None); ax.legend(fontsize=8)
    print(f"Weighted reflectance of the bare surface in n = {n_inc}: {100*Rw:.1f} %")
    return R, Rw


def escape_cone(n_semi, n_out=1.0):
    """Critical angle, escape-cone solid-angle fraction 1/(4n^2) (Yablonovitch 1982) and the 4n^2 path-length
    enhancement for a semiconductor of index n_semi emitting/trapping against a medium n_out."""
    nr = n_semi / n_out
    thc = np.degrees(np.arcsin(1 / nr)); frac = 1 / (4 * nr ** 2)
    T_normal = 1 - ((nr - 1) / (nr + 1)) ** 2
    print(f"n = {n_semi} into n = {n_out}: critical angle {thc:.1f}°, escape-cone fraction 1/(4n²) = {100*frac:.1f} %, "
          f"× (1−R_normal) = {100*frac*T_normal:.1f} % extracted per pass (isotropic source, single pass); 4n² = {4*nr**2:.0f}")
    return thc, frac


# ---------------- antireflection coatings
def design_slarc(substrate, lam0_nm, n_inc=1.0):
    """Ideal single-layer ARC at lam0: n_ARC = sqrt(n0 n_sub), d = lam0/(4 n_ARC). (Macleod Ch. 4.)"""
    ns = nk_of(substrate, [lam0_nm])[0].real; n0 = nk_of(n_inc, [lam0_nm])[0].real
    nA = np.sqrt(n0 * ns); d = lam0_nm / (4 * nA)
    print(f"Single-layer ARC at {lam0_nm} nm on n_sub = {ns:.2f} in n0 = {n0:.2f}: ideal n_ARC = {nA:.2f}, thickness {d:.0f} nm (quarter wave)")
    return nA, d

def design_dlarc(substrate, lam0_nm, n_inc=1.0):
    """Ideal quarter–quarter double-layer ARC: n_top = (n0^2 n_sub)^(1/3), n_bottom = (n0 n_sub^2)^(1/3). (Macleod Ch. 4.)"""
    ns = nk_of(substrate, [lam0_nm])[0].real; n0 = nk_of(n_inc, [lam0_nm])[0].real
    nt, nb = (n0 ** 2 * ns) ** (1 / 3), (n0 * ns ** 2) ** (1 / 3)
    print(f"Double-layer ARC at {lam0_nm} nm: ideal n_top = {nt:.2f} ({lam0_nm/(4*nt):.0f} nm), n_bottom = {nb:.2f} ({lam0_nm/(4*nb):.0f} nm)")
    return (nt, lam0_nm / (4 * nt)), (nb, lam0_nm / (4 * nb))

def quarter_wave(material, lam0_nm):
    """Thickness of a quarter-wave layer of this material at lam0."""
    n = nk_of(material, [lam0_nm])[0].real; d = lam0_nm / (4 * n)
    print(f"{material}: n({lam0_nm} nm) = {n:.3f} -> quarter-wave thickness {d:.1f} nm"); return d

def compare_stacks(stacks, substrate, lam_nm, n_inc=1.0, weight='am15g', theta0_deg=0.0, ax=None, title=None):
    """Plot R(lambda) for several coating stacks {label: [(material, d_nm), ...]} on the same substrate; print ⟨R⟩."""
    lam = np.asarray(lam_nm, float)
    if ax is None:
        fig, ax = plt.subplots(figsize=(7.2, 4.4))
    results = {}
    for label, layers in stacks.items():
        df = tmm(lam, n_inc, layers, substrate, theta0_deg)
        Rw = weighted_average(lam, df.R.values, weight); Aw = weighted_average(lam, df.Abs.values, weight)
        results[label] = (Rw, Aw)
        ax.plot(lam, 100 * df.R, lw=1.8, label=f"{label}: ⟨R⟩ = {100*Rw:.1f} %" + (f", ⟨A_film⟩ = {100*Aw:.1f} %" if Aw > 0.002 else ''))
    ax.set_xlabel('Wavelength (nm)'); ax.set_ylabel('Reflectance (%)'); ax.set_ylim(0, None); ax.legend(fontsize=8)
    ax.set_title(title or f'Coatings on {substrate if isinstance(substrate, str) else "substrate"}, incidence {theta0_deg:g}°, n0 = {n_inc}', fontsize=10)
    return results


# ---------------- rear reflectors and Bragg mirrors
def rear_reflector(semiconductor, dielectric, d_nm, metal, lam_nm, angles_deg=(0, 20, 40, 60, 80), ax=None):
    """Internal reflectance seen from inside the semiconductor at a rear stack semiconductor | dielectric (d) | metal,
    vs wavelength at normal incidence and vs angle at one wavelength (the latter is what textured light sees)."""
    lam = np.asarray(lam_nm, float)
    if ax is None:
        fig, axs = plt.subplots(1, 2, figsize=(12, 4.2))
    else:
        axs = ax
    for lab, layers in {'bare metal': [], f'{dielectric} {d_nm:g} nm / metal': [(dielectric, d_nm)]}.items():
        df = tmm(lam, semiconductor, layers, metal, 0.0)
        axs[0].plot(lam, 100 * df.R, lw=1.8, label=f"{lab}: R({lam[len(lam)//2]:.0f} nm) = {100*df.R.values[len(lam)//2]:.1f} %")
    axs[0].set_xlabel('Wavelength (nm)'); axs[0].set_ylabel('Internal Reflectance (%)'); axs[0].set_ylim(0, 101); axs[0].legend(fontsize=8)
    axs[0].set_title(f'Rear reflector, normal incidence, {metal}', fontsize=10)
    lam1 = lam[len(lam) // 2]; th = np.linspace(0, 89, 180)
    for lab, layers in {'bare metal': [], f'{dielectric} {d_nm:g} nm / metal': [(dielectric, d_nm)]}.items():
        R = [tmm([lam1], semiconductor, layers, metal, t).R.values[0] for t in th]
        axs[1].plot(th, 100 * np.array(R), lw=1.8, label=lab)
    nsemi = nk_of(semiconductor, [lam1])[0].real; nd = nk_of(dielectric, [lam1])[0].real
    if nd < nsemi:
        thc = np.degrees(np.arcsin(nd / nsemi)); axs[1].axvline(thc, color='0.5', ls=':', lw=1); axs[1].text(thc, 5, f' TIR beyond {thc:.0f}°', fontsize=8)
    axs[1].set_xlabel('Angle of Incidence Inside the Semiconductor (°)'); axs[1].set_ylabel(f'Internal Reflectance at {lam1:.0f} nm (%)'); axs[1].set_ylim(0, 101); axs[1].legend(fontsize=8)
    return axs

def dbr(n_high, n_low, pairs, lam0_nm, lam_nm, n_inc, n_sub, ax=None, label=None):
    """Quarter-wave distributed Bragg reflector of `pairs` (high/low) pairs centred at lam0. Materials may be constants
    or names. Prints the analytic peak reflectance R_max = [(1 - (nL/nH)^(2N) n_sub/n_inc) / (1 + ...)]^2 for constant n."""
    lam = np.asarray(lam_nm, float)
    nH = nk_of(n_high, [lam0_nm])[0].real; nL = nk_of(n_low, [lam0_nm])[0].real
    layers = [(n_high, lam0_nm / (4 * nH)), (n_low, lam0_nm / (4 * nL))] * pairs
    df = tmm(lam, n_inc, layers, n_sub, 0.0)
    if ax is None:
        fig, ax = plt.subplots(figsize=(6.8, 4.2))
    ax.plot(lam, 100 * df.R, lw=1.8, label=label or f'{pairs} pairs, n = {nH:.2f}/{nL:.2f}: peak R = {100*df.R.max():.2f} %')
    ax.set_xlabel('Wavelength (nm)'); ax.set_ylabel('Reflectance (%)'); ax.set_ylim(0, 101); ax.legend(fontsize=8)
    bw = 4 * lam0_nm / np.pi * np.arcsin((nH - nL) / (nH + nL))
    print(f"DBR {pairs} × ({nH:.2f}/{nL:.2f}) quarter waves at {lam0_nm} nm: peak R = {100*df.R.max():.2f} % at {lam[df.R.values.argmax()]:.0f} nm; "
          f"stop band ≈ {bw:.0f} nm wide (Macleod: Δλ ≈ (4λ0/π) arcsin[(nH − nL)/(nH + nL)])")
    return df

print('Module C loaded: nk_of, fresnel_normal, tmm, bare_surface_reflectance, escape_cone, design_slarc, design_dlarc, quarter_wave, compare_stacks, rear_reflector, dbr. Coating materials:', ', '.join(COATING_SOURCES))


In [ ]:
# YOUR CODE HERE (Task 3.1) — your material; set the wavelength range to its absorption range (or the detection band)
lam = np.arange(300, 1201, 5.0)                     # nm; for a 1.1 eV absorber; use e.g. 300–900 for CdTe, 300–1700 for InGaAs
mat = load_nk('Si')                                  # or load_nk_csv('my_nk.csv'); or any key of NK_SOURCES / COATING_SOURCES
fig, ax = plt.subplots(figsize=(6.8, 4.2))
R_air, Rw_air = bare_surface_reflectance(mat, lam, n_inc=1.0, weight='am15g', label='Si in air', ax=ax)
R_eva, Rw_eva = bare_surface_reflectance(mat, lam, n_inc=1.5, weight='am15g', label='Si under encapsulant (n = 1.5)', ax=ax)
fig.savefig('bare_reflectance.png', dpi=200, bbox_inches='tight')

# Photodetectors: R at the detection wavelength
# print(float(np.interp(1550, lam, R_air)))
# Emitters: escape cone of your material against air and encapsulant
# escape_cone(n_semi=2.45, n_out=1.0); escape_cone(n_semi=2.45, n_out=1.5)


**Task 3.1 Answer** *(8 points)*

**Data set and reference; $\langle R \rangle$ in air and under encapsulant (or $R$ at the detection wavelength; or critical angle and extraction fraction in air and encapsulant):** *(...)*

**Which wavelengths dominate the loss and why (dispersion of $n$, where the photon flux is):** *(two or three sentences)*


### 3.2 Surfaces in your corpus: texture and light management (12 points)

Identify from your corpus the **surface** approaches used to reduce optical loss, as distinct from thin-film coatings: alkaline or acidic texturing and black silicon for wafers; textured TCOs and substrates, roughening, patterned sapphire substrates, die shaping, photonic crystals or microlenses for thin films and emitters; cleaved or etched facets for lasers; and the encapsulant or dome. For each, record the process (etchant and temperature, etch time, mask, lithography or stamping, growth on a patterned template) and the structure it produces (pyramid size and facet angle, roughness, feature pitch relative to the wavelength), and the performance metric the paper uses (weighted reflectance, $J_{sc}$ gain, light-extraction efficiency, EQE). Add the columns `front_texture` (the structure) and `texture_process` (how it was made) to `survey.csv`.

**Write the surfaces part of the Optical Surfaces and Coatings subsection** (8 points, report): the texturing or light-management structures used, their processing, the physics by which they work (double bounce and oblique coupling, index grading, escape-cone widening, randomization toward the $4n^2$ limit), and the measured gain, with the bare-surface numbers of Task 3.1 as the baseline.


**Task 3.2 Answer** *(4 points)*

**Surface structures and processes in your corpus (structure — process — metric — bibtex key):** *(...)*

**Report status checklist:**

* ☐ Texture / light-management structures described with their processing
* ☐ Mechanism explained (double bounce, grading, escape cone, randomization)
* ☐ Gain quantified against the bare-surface baseline


---
## Task 4 — Coatings: antireflection coatings and reflectors (30 points + 5 bonus)

Completes the **Optical Surfaces and Coatings** subsection. Points: 4.1 — 10 (notebook); 4.2 — 8 (notebook); 4.3 — 12 (4 validation/notebook, 8 report); 4.4 — bonus 5.

### Background: thin-film interference

A thin film adds a second reflecting interface, and the two reflected waves interfere; at the quarter-wave thickness $n_1 d = \lambda_0/4$ they are $\pi$ out of phase and the reflectance at $\lambda_0$ is minimized. The minimum reaches zero when $n_1 = \sqrt{n_0 n_{sub}}$; for two quarter-wave layers the zero-reflectance condition is $n_{top} = (n_0^2 n_{sub})^{1/3}$ and $n_{bottom} = (n_0 n_{sub}^2)^{1/3}$, which gives a broader, flatter minimum (Macleod, Ch. 4). Because $n_{sub}$ and the ideal indices depend on the incident medium, a coating designed for air is no longer optimal under an encapsulant. A reflector is the same physics run the other way: a low-index spacer between the semiconductor and a metal raises the internal reflectance and, beyond the critical angle, turns it into total internal reflection; a stack of alternating quarter-wave high- and low-index layers (a distributed Bragg reflector) reaches reflectances above 99 % over a stop band of width $\Delta\lambda \approx (4\lambda_0/\pi)\arcsin[(n_H - n_L)/(n_H + n_L)]$ (Macleod, Ch. 5). The transfer-matrix method in the module computes $R(\lambda)$ for any stack from the characteristic matrices of its layers, including absorption in the films.

### 4.1 Single- and double-layer ARC design for your device (10 points)

Choose the design wavelength and the incident medium for *your* device — near 600 nm in air or under encapsulant ($n_0 = 1.5$) for a silicon or thin-film solar cell; the signal wavelength for a photodetector; the emission wavelength for an ARC on the window layer of an LED; the lasing wavelength for a low-reflectance facet — and:

1. compute the ideal $n_{ARC}$ and thickness with `design_slarc`, and the ideal pair with `design_dlarc`;
2. choose *real* coating materials whose $n$ at $\lambda_0$ comes closest (the module carries SiN$_x$ of two compositions, SiO$_2$, Al$_2$O$_3$, MgF$_2$, TiO$_2$, Ta$_2$O$_5$, ZnS, ITO; add others with `load_nk`), set their quarter-wave thicknesses with `quarter_wave`, and compute $R(\lambda)$ and the weighted $\langle R \rangle$ for the bare surface, the single layer and the double layer with `compare_stacks`;
3. state what $\kappa$ the ARC must have and read the film absorptance $\langle A_{film} \rangle$ that the module prints — a coating that absorbs the blue is a loss too.

Then explain, in the answer cell, how the choice of design wavelength and index follows from your device (where the photon flux or the emission sits, what the incident medium is) and what the double layer buys over the single layer.


In [ ]:
# YOUR CODE HERE (Task 4.1) — design wavelength, incident medium and substrate for YOUR device
lam0, n0 = 600, 1.0                                  # e.g. 600 nm in air; 1.5 under encapsulant; 1550 nm for a telecom detector
sub = mat                                            # the n,k loaded in Task 3.1
nA, dA = design_slarc(sub, lam0, n_inc=n0)
(nt, dt), (nb, db) = design_dlarc(sub, lam0, n_inc=n0)

# real materials closest to the ideal indices (edit):
d_sinx = quarter_wave('SiNx (PECVD, n~2.1)', lam0)
d_sio2 = quarter_wave('SiO2', lam0); d_tio2 = quarter_wave('TiO2 (film)', lam0)
fig, ax = plt.subplots(figsize=(7.2, 4.5))
res = compare_stacks({
    'bare': [],
    f'ideal single layer n = {nA:.2f}': [(nA, dA)],
    f'SiNx {d_sinx:.0f} nm': [('SiNx (PECVD, n~2.1)', d_sinx)],
    f'SiO2 {d_sio2:.0f} / TiO2 {d_tio2:.0f} nm': [('SiO2', d_sio2), ('TiO2 (film)', d_tio2)],
}, sub, lam, n_inc=n0, weight='am15g', ax=ax, title=f'ARC designs on Si, {lam0:g} nm design wavelength, n0 = {n0}')   # weight: 'am15g' (solar), or emission_weight(lam, peak, fwhm) for an LED
fig.savefig('arc_design.png', dpi=200, bbox_inches='tight')


**Task 4.1 Answer** *(10 points)*

**Design wavelength and incident medium, and why:** *(...)*

**Ideal $n$ and $d$ for the single and double layers; the real materials chosen and their $n$, $\kappa$ at $\lambda_0$:** *(...)*

| stack | $\langle R \rangle$ (%) | $\langle A_{film} \rangle$ (%) |
|---|---|---|
| bare | ... | ... |
| single layer | ... | ... |
| double layer | ... | ... |

**What the double layer buys, and what the coating's absorption costs:** *(two or three sentences)*


### 4.2 The reflector (8 points)

Every device has a second optical surface. For a solar cell it is the rear: a dielectric spacer (SiO$_2$, AlO$_x$) between the silicon and the metal raises the internal reflectance from ~85 % (bare Al) toward 97–99 % and makes the weakly absorbed near-infrared light take a second pass, and textured light arrives obliquely, so the angle matters as much as the wavelength. For an LED it is a mirror under the active region (a metal, a DBR, or an omnidirectional reflector of dielectric and metal) that sends the downward half of the emission back toward the escape cone; for a VCSEL or a resonant-cavity detector it is a DBR of 20–40 pairs; for an edge-emitting laser it is the high-reflectance facet coating. Compute the one that applies to your device:

* **Solar cells and photodetectors:** `rear_reflector(semiconductor, dielectric, d_nm, metal, lam)` — internal reflectance vs. wavelength at normal incidence and vs. angle at one wavelength, for the bare metal and for the dielectric/metal stack. Vary the spacer thickness and report the gain at the near-band-edge wavelength and the angle beyond which total internal reflection takes over.
* **Emitters:** `dbr(n_high, n_low, pairs, lam0, lam, n_inc, n_sub)` — reflectance of a quarter-wave stack for the material pair your corpus uses (AlGaAs/GaAs, AlGaN/GaN, TiO$_2$/SiO$_2$ ...); report the peak reflectance and stop-band width versus the number of pairs, and compare with a metal mirror (`compare_stacks` with the metal as substrate, seen from inside the semiconductor).


In [ ]:
# YOUR CODE HERE (Task 4.2) — the reflector of your device
# Solar cells / photodetectors: rear dielectric + metal, seen from inside the absorber
lam_ir = np.arange(900, 1201, 5.0)                   # near the band edge of silicon; adjust for your absorber
axs = rear_reflector(mat, 'SiO2', 150, 'Al', lam_ir)
plt.savefig('rear_reflector.png', dpi=200, bbox_inches='tight')

# Emitters: DBR for your material pair (constants or loaded n,k), e.g. 20 pairs of AlAs/GaAs-like indices at 980 nm
# fig, ax = plt.subplots(); dbr(3.5, 3.0, 20, 980, np.arange(850, 1121, 1.0), n_inc=3.5, n_sub=3.5, ax=ax)
# fig, ax = plt.subplots(); dbr('TiO2 (film)', 'SiO2', 6, 450, np.arange(350, 600, 1.0), n_inc=1.0, n_sub='GaN', ax=ax)


**Task 4.2 Answer** *(8 points)*

**Reflector type and stack for your device, and the computed reflectance (vs. spacer thickness and angle, or vs. number of pairs):** *(...)*

**What the reflector buys the device (second-pass absorption near the band edge; extraction for an emitter) and what limits it (parasitic absorption in the metal, angular range of the DBR):** *(...)*


### 4.3 Coatings in your corpus and the narrative (12 points)

Add the remaining columns to `survey.csv` for your top-20 rows and re-run the validation cell under Task 2.3:

| column | content |
|---|---|
| `arc_stack` | the antireflection or extraction coating(s) with thicknesses (e.g. `SiNx 75 nm`; `SiO2/SiNx DLARC`; `MgF2/ZnS`; `none`) |
| `arc_process` | deposition and conditions (e.g. `PECVD SiH4/NH3 400 C`; `e-beam evaporation`; `ALD`) |
| `rear_reflector_stack` | the reflector (e.g. `AlOx 10 nm / SiNx 100 nm / Al`; `Ag`; `20-pair AlGaAs DBR`; `cleaved facet + HR coating`) |
| `other_optical` | anything else used to manage light (`encapsulant dome`, `phosphor`, `PSS`, `photonic crystal`, `none`) |
| `optical_metric` | the optical figure the paper reports (`<R> = 2.1 %`; `Jsc gain 1.2 mA/cm2`; `LEE 80 %`; `EQE 90 % at 1550 nm`), or `not reported` |

**Write the coatings part of the Optical Surfaces and Coatings subsection** (8 points, report): the ARC and reflector designs used in your field and how they are deposited, the film structure and optical constants they rely on ($n$, $\kappa$, thickness control, stability under the later process steps), the reflectance or extraction they achieve, and the impact on $J_{sc}$, EQE or light-extraction efficiency — with your own Task 4.1–4.2 designs as the point of comparison: do the papers use the indices and thicknesses your calculation says they should, and if not, why (processing constraints, encapsulation, dual roles such as passivation and hydrogen supply)?


**Task 4.3 Answer** *(4 points)*

**Dominant ARC and reflector designs in your corpus, with processes and metrics (bibtex keys):** *(...)*

**Where the published designs differ from your calculation, and why:** *(two or three sentences)*

**Report status checklist:**

* ☐ ARC and reflector designs described with deposition processes
* ☐ Optical constants and thicknesses discussed against the calculated optimum
* ☐ Performance impact quantified ($J_{sc}$, EQE, extraction efficiency)
* ☐ Gaps in the corpus named explicitly


### 4.4 Bonus — an optical loss budget from measured data (up to 5 points)

If an article in your corpus (or a datasheet) gives a measured reflectance spectrum $R(\lambda)$ or an EQE curve for your device, digitize it (WebPlotDigitizer or similar), load it as a CSV, and convert each loss into a current: $J_{loss} = q\int \Phi_{AM1.5G}(\lambda)\,R(\lambda)\,d\lambda$ for the front reflectance (use `am15g_photon_weight` and `weighted_average`), and the same integral of $1 - EQE(\lambda)$ over the absorption range for the total loss; for an LED use the reported light-extraction efficiency instead. Compare the reflectance loss with your Task 4.1 design and attribute the remainder (escape reflectance, parasitic absorption in the ARC and TCO, recombination). Put the budget in the report as a short table with the source cited.


**Task 4.4 Answer (bonus)**

**Source of the measured curve; $J_{loss}$ from reflectance and from $1 - EQE$ (or LEE); attribution of the remainder:** *(...)*


---
## AI Use Disclosure

Per the course AI policy, disclose specifically where and how generative AI was used in completing this assignment (tool, purpose, and which tasks) — including Gemini in Colab for code and any tool used to search or summarize literature. If you did not use AI, state that.

*Your disclosure here (double-click to edit).*


## Submission Checklist

1. **Runtime → Restart and run all** — the notebook runs top to bottom with no errors (the survey validation cell must print `Validation PASSED`). The runtime is ephemeral: re-upload `survey.csv` (and any n,k or digitized CSV) after a restart.
2. All `# YOUR CODE HERE` cells are implemented for *your* material and device, and all *Your answer here* prompts are answered.
3. The AI Use Disclosure above is completed.
4. **File → Print → Save as PDF** — `EMA-6611-HW03-LastName.pdf`. Check that all figures are visible.
5. **File → Download → Download .ipynb** — the notebook file itself.
6. In Overleaf: **Recompile**, check for `[?]` and overfull tables, then **Download PDF** — `LastName-Report-HW03.pdf`.
7. Submit **four files** (notebook PDF, .ipynb, extended survey.csv, report PDF) on Webcourses.
8. Keep your copy in Drive — Homework 4 (Metallization, Interconnection and Packaging) adds columns to the same survey.csv and the next report subsections.

## References cited in this assignment

* Aberle, A. G., Glunz, S., Warta, W. (1992). Impact of illumination level and oxide parameters on Shockley–Read–Hall recombination at the Si–SiO$_2$ interface. *J. Appl. Phys.* 71, 4422–4431.
* Cuevas, A., Wan, Y., Yan, D., Samundsett, C., Allen, T., Zhang, X., Cui, J., Bullock, J. (2018). Carrier population control and surface passivation in solar cells. *Sol. Energy Mater. Sol. Cells* 184, 38–47.
* Dingemans, G., Kessels, W. M. M. (2012). Status and prospects of Al$_2$O$_3$-based surface passivation schemes for silicon solar cells. *J. Vac. Sci. Technol. A* 30, 040802.
* Girisch, R. B. M., Mertens, R. P., De Keersmaecker, R. F. (1988). Determination of Si–SiO$_2$ interface recombination parameters using a gate-controlled point-junction diode under illumination. *IEEE Trans. Electron Devices* 35, 203–222.
* Grundmann, M. (2021). *The Physics of Semiconductors*, 4th ed. Springer. Ch. 9, 10, 23 and 24.
* Hall, R. N. (1952). Electron-hole recombination in germanium. *Phys. Rev.* 87, 387.
* Johnston, M. B., Herz, L. M. (2016). Hybrid perovskites for photovoltaics: charge-carrier recombination, diffusion, and radiative efficiencies. *Acc. Chem. Res.* 49, 146–154.
* Kane, D. E., Swanson, R. M. (1985). Measurement of the emitter saturation current by a contactless photoconductivity decay method. *Proc. 18th IEEE Photovoltaic Specialists Conf.*, 578–583.
* Macleod, H. A. (2001). *Thin-Film Optical Filters*, 3rd ed. Institute of Physics Publishing. Ch. 2, 4, 5.
* Piprek, J. (2010). Efficiency droop in nitride-based light-emitting diodes. *Phys. Status Solidi A* 207, 2217–2225.
* Polyanskiy, M. N. (2024). Refractiveindex.info database of optical constants. *Sci. Data* 11, 94.
* Richter, A., Glunz, S. W., Werner, F., Schmidt, J., Cuevas, A. (2012). Improved quantitative description of Auger recombination in crystalline silicon. *Phys. Rev. B* 86, 165202.
* Shockley, W., Read, W. T. (1952). Statistics of the recombinations of holes and electrons. *Phys. Rev.* 87, 835–842.
* Sproul, A. B. (1994). Dimensionless solution of the equation describing the effect of surface recombination on carrier decay in semiconductors. *J. Appl. Phys.* 76, 2851–2854.
* Trupke, T., Green, M. A., Würfel, P., Altermatt, P. P., Wang, A., Zhao, J., Corkish, R. (2003). Temperature dependence of the radiative recombination coefficient of intrinsic crystalline silicon. *J. Appl. Phys.* 94, 4930–4937.
* Wong, M. S., Hwang, D., Alhassan, A. I., Lee, C., Ley, R., Nakamura, S., DenBaars, S. P. (2018). High efficiency of III-nitride micro-light-emitting diodes by sidewall passivation using atomic layer deposition. *Opt. Express* 26, 21324–21331.
* Yablonovitch, E. (1982). Statistical ray optics. *J. Opt. Soc. Am.* 72, 899–907.
* ASTM G173-03, reference solar spectra, via pvlib (Anderson, K. S. et al., *J. Open Source Softw.* 8, 5994 (2023)).
